In [ ]:
#Loading packages
!pip install -q ISLP    #install this ISLP package
from ISLP import load_data
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from ISLP.models import ModelSpec as MS
from sklearn.pipeline import Pipeline
import statsmodels.api as sm

#Loading Hitters dataset
Hitters = load_data('Hitters').dropna()
display(Hitters.shape)  # dimension is 322x20
display(Hitters.head())
display(Hitters.dtypes)  # mix of floating/integer and categorical variables


#Preparing response and predictor variables
y = np.array(Hitters['Salary'])
#this ISLP package command encodes categorical variables similar to pd.get_dummies
X = MS(Hitters.columns.drop('Salary')).fit_transform(Hitters)


# --- Lasso regression with CV ---
pipe = Pipeline([('scaler', StandardScaler()), ('lasso', LassoCV())])
pipe.fit(X, y)
tuned_lasso = pipe.named_steps['lasso']
lasso_coef = pd.Series(tuned_lasso.coef_, index=X.columns)

# --- Ridge regression with CV ---
pipe = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV())])
pipe.fit(X, y)
tuned_ridge = pipe.named_steps['ridge']
ridge_coef = pd.Series(tuned_ridge.coef_, index=X.columns)

# --- OLS ---
ols = Pipeline([
    ('scaler', StandardScaler()),
    ('ols', LinearRegression())
]).fit(X, y)
ols_coef = pd.Series(ols.named_steps['ols'].coef_, index=X.columns)

# --- Combine results ---
coef_table = pd.concat(
    [ols_coef, ridge_coef, lasso_coef],
    axis=1,
    keys=['OLS', 'Ridge', 'Lasso']
).round(3)

# Add best λ row (use tuned_* not the pipeline itself!)
coef_table.loc['Best λ'] = [np.nan, tuned_ridge.alpha_, tuned_lasso.alpha_]

print(coef_table.to_string())





  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 9.9 MB/s eta 0:00:00


(263, 20)

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,League,Division,PutOuts,Assists,Errors,Salary,NewLeague
1,315,81,7,24,38,39,14,3449,835,69,321,414,375,N,W,632,43,10,475.0,N
2,479,130,18,66,72,76,3,1624,457,63,224,266,263,A,W,880,82,14,480.0,A
3,496,141,20,65,78,37,11,5628,1575,225,828,838,354,N,E,200,11,3,500.0,N
4,321,87,10,39,42,30,2,396,101,12,48,46,33,N,E,805,40,4,91.5,N
5,594,169,4,74,51,35,11,4408,1133,19,501,336,194,A,W,282,421,25,750.0,A


,0
AtBat,int64
Hits,int64
HmRun,int64
Runs,int64
RBI,int64
Walks,int64
Years,int64
CAtBat,int64
CHits,int64
CHmRun,int64


                  OLS    Ridge       Lasso
intercept       0.000    0.000    0.000000
AtBat        -291.095 -270.475 -226.830000
Hits          337.830  296.562  254.827000
HmRun          37.854   18.164    0.000000
Runs          -60.572  -29.405   -0.000000
RBI           -26.995   -9.216    0.000000
Walks         135.074  124.440  102.145000
Years         -16.693  -38.692  -44.594000
CAtBat       -391.039 -225.357   -0.000000
CHits          86.688  126.900    0.000000
CHmRun        -14.182   39.080   43.363000
CRuns         480.747  320.210  218.029000
CRBI          260.690  160.370  123.421000
CWalks       -213.892 -184.494 -138.617000
League[N]      31.249   31.019   16.041000
Division[W]   -58.414  -60.246  -59.545000
PutOuts        78.761   78.610   76.104000
Assists        53.732   47.446   24.744000
Errors        -22.161  -23.764  -13.253000
NewLeague[N]  -12.349  -13.674   -0.000000
Best λ            NaN    1.000    2.737306


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# --- Simulation setup ---
np.random.seed(123)
n = 100

X1 = np.random.normal(50, 10, size=n)   # large scale
X2 = np.random.normal(5, 1, size=n)     # small scale
X = pd.DataFrame({'X1': X1, 'X2': X2})

y = 3*X1 - 2*X2 + np.random.normal(0, 5, n)

print("ORIGINAL X (first 5 rows):")
print(X.head(), "\n")

print("ORIGINAL y (first 5):")
print(y[:5], "\n")

# --- Raw OLS ---
ols_raw = LinearRegression().fit(X, y)
ols_raw_coef = pd.Series(ols_raw.coef_, index=X.columns)

# --- Pipeline (StandardScaler + OLS) ---
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ols', LinearRegression())
]).fit(X, y)

scaler = pipe.named_steps['scaler']
ols_scaled = pipe.named_steps['ols']

# Back-transform coefficients
ols_coef_back = pd.Series(
    ols_scaled.coef_ / scaler.scale_,
    index=X.columns
)

# --- Show comparison ---
coef_compare = pd.DataFrame({
    "OLS_raw": ols_raw_coef,
    "OLS_pipeline_back": ols_coef_back
})

print("First 5 rows of scaled X:")
print(pd.DataFrame(scaler.transform(X), columns=X.columns).head(), "\n")

print("Coefficient comparison:")
print(coef_compare.round(3))

ORIGINAL X (first 5 rows):
          X1        X2
0  39.143694  5.642055
1  59.973454  3.022112
2  52.829785  5.712265
3  34.937053  7.598304
4  44.213997  4.975374 

ORIGINAL y (first 5):
[109.66352311 170.8856126  158.06833617  93.05603538 122.65970815] 

First 5 rows of scaled X:
         X1        X2
0 -0.986261  0.682014
1  0.859955 -2.018808
2  0.226786  0.754392
3 -1.359111  2.698655
4 -0.536862 -0.005248 

Coefficient comparison:
    OLS_raw  OLS_pipeline_back
X1    2.974              2.974
X2   -1.886             -1.886


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# --- Simulation setup ---
np.random.seed(123)
n, p = 100, 200
beta_true = np.zeros(p)
beta_true[:5] = [5, -3, 1, 4, -2]  # sparse signal

X = np.random.normal(size=(n, p))
y = X @ beta_true + np.random.normal(scale=3, size=n)

# --- Train/test split (30%) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123
)

# --- OLS (no scaling needed) ---
ols = LinearRegression().fit(X_train, y_train)

# --- Ridge with scaling inside pipeline ---
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', RidgeCV())
]).fit(X_train, y_train)
ridge_alpha = ridge_pipe.named_steps['ridge'].alpha_

# --- Lasso with scaling inside pipeline ---
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV())
]).fit(X_train, y_train)
lasso_alpha = lasso_pipe.named_steps['lasso'].alpha_

# --- Coefficients ---
coef_compare = pd.DataFrame({
    "Truth": beta_true,
    "OLS": ols.coef_,
    "Ridge": ridge_pipe.named_steps['ridge'].coef_,
    "Lasso": lasso_pipe.named_steps['lasso'].coef_
})

print("Coefficient comparison (first 10 predictors):")
print(coef_compare.round(2).head(10), "\n")

# --- Test MSEs ---
ols_mse = mean_squared_error(y_test, ols.predict(X_test))
ridge_mse = mean_squared_error(y_test, ridge_pipe.predict(X_test))
lasso_mse = mean_squared_error(y_test, lasso_pipe.predict(X_test))

print("Test MSEs:")
print(f"OLS: {ols_mse:.2f}, Ridge: {ridge_mse:.2f}, Lasso: {lasso_mse:.2f}\n")

# --- Report chosen alphas ---
print("Chosen Ridge alpha:", ridge_alpha)
print("Chosen Lasso alpha:", lasso_alpha)

Coefficient comparison (first 10 predictors):
   Truth   OLS  Ridge  Lasso
0    5.0  1.73   1.55   4.50
1   -3.0 -0.81  -0.72  -1.68
2    1.0  0.81   0.69   0.72
3    4.0  1.47   1.47   3.61
4   -2.0 -0.69  -0.56  -1.32
5    0.0  0.07   0.07   0.14
6    0.0 -0.27  -0.26  -0.00
7    0.0 -0.61  -0.52  -0.00
8    0.0  0.35   0.36   0.00
9    0.0 -0.21  -0.21  -0.00 

Test MSEs:
OLS: 49.99, Ridge: 44.65, Lasso: 11.03

Chosen Ridge alpha: 10.0
Chosen Lasso alpha: 0.4269777413606456
